# Construcción de un meta clasificador para las 2 capas de análisis

In [6]:
from google.colab import drive
drive.mount('/content/drive')
LGBM_MODEL_PATH = '/content/drive/MyDrive/TFG_Posdata/models/lgbm_v1/model.pkl'
LGBM_CONFIG_PATH = '/content/drive/MyDrive/TFG_Posdata/models/lgbm_v1/config.json'
DISTILBERT_MODEL_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1'
DISTILBERT_TOKENIZER_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1'
DISTILBERT_CONFIG_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1/own_config.json'
DATASET_PATH = '/content/drive/MyDrive/TFG_Posdata/datasets/dataset_final.csv'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import pandas as pd
import numpy as np
import sklearn
import json
import joblib
import torch
import transformers
import scipy
from sklearn.model_selection import train_test_split

lgbm_model = joblib.load(LGBM_MODEL_PATH)
lgbm_config = json.load(open(LGBM_CONFIG_PATH))

distilbert_model = transformers.AutoModelForSequenceClassification.from_pretrained(DISTILBERT_MODEL_PATH)
distilbert_tokenizer = transformers.AutoTokenizer.from_pretrained(DISTILBERT_TOKENIZER_PATH)
distilbert_config = json.load(open(DISTILBERT_CONFIG_PATH))

df = pd.read_csv(DATASET_PATH)

X = df['text']
y = (df['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [9]:
import re

def predict_distilbert(texts, batch_size=32):
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = distilbert_tokenizer(
            batch,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = distilbert_model(**inputs).logits

        probs = torch.nn.functional.softmax(logits, dim=1)
        all_probs.append(probs[:, 1].cpu().numpy())

    return np.concatenate(all_probs)

urgency_words = ['urgente','urgently','urgent','inmediatamente','immediately',
                   'ahora','now','hoy','today','bloquea','blocked','suspendida',
                   'suspended','cancel','cancela','verifique','verify']
action_words = ['haga clic','click','acceda','access','llame','call','responda',
                   'reply','confirme','confirm','descargue','download','ingrese','enter']
financial_words = ['cuenta','account','banco','bank','tarjeta','card','pago','payment',
                   'transferencia','transfer','bizum','credito','credit','débito','debit']
prize_words = ['gratis','free','premio','prize','ganador','winner','regalo','gift',
                   'oferta','offer','descuento','discount','gana','win']
threat_words = ['amenaza', 'threat', 'peligro', 'danger', 'dangerous', 'peligroso', 'cuidado',
                'beware', 'attention', 'atencion', 'careful', 'cuidado']
impersonation_words = [
    # Bancos
    "santander", "bbva", "caixabank", "bankinter", "sabadell",
    "bankia", "ing", "kutxabank", "ibercaja", "unicaja",
    "abanca", "cajamar", "openbank", "bizum",
    # Operadoras
    "movistar", "vodafone", "orange", "masmovil", "yoigo",
    "jazztel", "lowi",
    # Envíos
    "correos", "seur", "mrw", "dhl", "fedex", "ups", "gls", "celeritas",
    # Instituciones
    "hacienda", "tributaria", "dgt", "sepe", "ministerio",
    "ayuntamiento", "policia", "guardia civil", "seguridad social",
    # Servicios digitales
    "apple", "google", "microsoft", "netflix", "spotify",
    "whatsapp", "facebook", "instagram", "icloud",
    # Compras
    "mercadona", "lidl", "carrefour", "alcampo", "aldi",
    "zara", "inditex", "repsol", "bp", "iberdrola", "endesa", "naturgy",
]

_http_pattern = r"https?://[^\s]+"
_www_pattern = r"www\.[^\s]+"
url_pattern     = re.compile(f"({_http_pattern})|({_www_pattern})")
shortener_pattern = re.compile(r'\b(bit\.ly|t\.co|tinyurl\.com|goo\.gl|ow\.ly|rb\.gy|cutt\.ly)\b')
phone_pattern   = re.compile(r'(\+?[1-9]\d{1,14}|[0-9]{9,15})')

def extract_features (df):
  numeric_df = pd.DataFrame()

  #Features estructurales
  numeric_df['text_len'] = df['text'].str.len()
  numeric_df['word_count'] = df['text'].str.split().str.len()
  numeric_df['avg_word_len'] = numeric_df['text_len'] / numeric_df['word_count'].replace(0, 1)
  numeric_df['caps_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper())/max(len(x), 1))
  numeric_df['digit_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isdigit())/max(len(x), 1))
  numeric_df['excl_count'] = df['text'].str.count('!')
  numeric_df['ques_count'] = df['text'].str.count(r'\?')
  numeric_df['num_count'] = df['text'].str.count(r'\d')

  #Features semánticas
  numeric_df['has_urgency'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in urgency_words))
  numeric_df['has_action'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in action_words))
  numeric_df['has_financial'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in financial_words))
  numeric_df['has_prize'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in prize_words))
  numeric_df['has_threat'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in threat_words))
  numeric_df['has_impersonation'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in impersonation_words))

  #Features de contenido
  numeric_df['has_url'] = df['text'].apply(lambda t: bool(url_pattern.search(str(t))))
  numeric_df['has_phone'] = df['text'].apply(lambda t: bool(phone_pattern.search(str(t))))
  numeric_df['url_len'] = df['text'].apply(lambda t: (m := url_pattern.search(str(t))) and len(m.group(0)) or 0)
  numeric_df['has_shortener'] = df['text'].apply(lambda t: bool(shortener_pattern.search(str(t))))

  return numeric_df

In [12]:
X_test_df = X_test.reset_index(drop=True).to_frame(name='text')
X_test_features = extract_features(X_test_df)

scores_lgbm = lgbm_model.predict_proba(X_test_features)[:, 1]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
distilbert_model.to(device)
distilbert_model.eval()
scores_distilbert = predict_distilbert(X_test.tolist())

X_meta = np.column_stack((scores_lgbm, scores_distilbert))

In [13]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(class_weight='balanced', random_state=42)
meta_model.fit(X_meta, y_test)

LogisticRegression(class_weight='balanced', random_state=42)

In [15]:
from sklearn.metrics import f1_score

y_pred_meta = meta_model.predict(X_meta)
y_scores_meta = meta_model.predict_proba(X_meta)[:, 1]

print(f"Ensemble:      F1 = {f1_score(y_test, y_pred_meta, average='weighted'):.4f}")

Ensemble:      F1 = 0.9895


In [18]:
import joblib
import json

from google.colab import drive
drive.mount('/content/drive')
ruta_carpeta = '/content/drive/MyDrive/TFG_Posdata/models/meta_v1/'

nombre_modelo = 'model.pkl'
with open(ruta_carpeta + nombre_modelo, 'wb') as f:
  joblib.dump(meta_model, f)
print("Modelo guardado con éxito")

nombre_config = 'config.json'
config = {
    "model_version": "meta_v1",
    "features": ["score_lgbm", "score_distilbert"],
    "threshold": 0.5
}
with open(ruta_carpeta + nombre_config, 'w') as f:
    json.dump(config, f)
print("Configuración guardada con éxito")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo guardado con éxito
Configuración guardada con éxito
